In [1]:
import torch
from torchinfo import summary
from improved_diffusion.script_util import sr_create_model_and_diffusion

In [8]:
dict = {"in_channel":1,
        "dims":2,
        "num_channels":96,
        "num_res_blocks":2,
        "num_heads":4,
        "num_heads_upsample":-1,
        "attention_resolutions":"2,4,8",
        "dropout":0.0,
        "learn_sigma":False,
        "class_cond":False,
        "diffusion_steps":4000,
        "noise_schedule":"linear",
        "timestep_respacing":"",
        "use_kl":False,
        "predict_xstart":False,
        "rescale_timesteps":True,
        "rescale_learned_sigmas":True,
        "use_checkpoint":False,
        "use_scale_shift_norm":True,
        "spatial_size":128,
        }

In [9]:
model, diffusion = sr_create_model_and_diffusion(
    **dict
)

(64, 32, 16)


In [11]:
class_cond = dict["class_cond"]
diffusion_steps = 2


N, C = 1, 1
ndims = dict["dims"]
spatial = 64

# create tuple of spatial dimensions
high_shape = (spatial * 4,) * ndims
low_shape  = (spatial,) * ndims

x = torch.rand(N, C, *high_shape)
low_res = torch.rand(N, C, *low_shape)


t = torch.randint(0, diffusion_steps, (N,), dtype=torch.long)

if class_cond:
    y = torch.zeros(N, dtype=torch.long)  # or random labels
    out = model(x, t, low_res=low_res, y=y)
    summary(
        model,
        input_data=(x, t, low_res, y),
        verbose=1,
        depth = 3,
        device="cpu",   # or "cuda"
    )
else:
    out = model(x, t, low_res=low_res)
    summary(
        model,
        input_data=(x, t, low_res),
        verbose=1,
        depth = 3,
        device="cpu",   # or "cuda"
    )


print(out.shape)

Layer (type:depth-idx)                        Output Shape              Param #
SuperResModel                                 [1, 1, 256, 256]          --
├─Sequential: 1-1                             [1, 384]                  --
│    └─Linear: 2-1                            [1, 384]                  37,248
│    └─SiLU: 2-2                              [1, 384]                  --
│    └─Linear: 2-3                            [1, 384]                  147,840
├─ModuleList: 1-2                             --                        --
│    └─TimestepEmbedSequential: 2-4           [1, 96, 256, 256]         --
│    │    └─Conv2d: 3-1                       [1, 96, 256, 256]         1,824
│    └─TimestepEmbedSequential: 2-5           [1, 96, 256, 256]         --
│    │    └─ResBlock: 3-2                     [1, 96, 256, 256]         240,384
│    └─TimestepEmbedSequential: 2-6           [1, 96, 256, 256]         --
│    │    └─ResBlock: 3-3                     [1, 96, 256, 256]         240,38